In [1]:
## Import modules
import os, sys
import numpy as np
import geopandas as gpd
import cftime
import gc
import shapely
import json
from ipywidgets import interact, FloatSlider
import ipywidgets as widgets
import matplotlib.pyplot as plt
import glob
# Import Plotly for interactive plotting
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.colors as pc
from ipywidgets import interact, IntSlider, Dropdown, VBox, HBox
import ipywidgets as widgets

# Add the directory containing 'cmct' to the Python path
# Navigate two levels up to reach main CmCt dir
cmct_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir, os.pardir))

# Initialising Logger
import logging

logging.basicConfig(
    level=logging.ERROR, format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

# Import utilities for this comparison
sys.path.insert(0, cmct_dir)
from cmct.time_utils import *
from cmct.calving import *
from cmct.calving_modules.interpolation import *
from cmct.calving_modules.residual_calculation import *
# from cmct.calving_modules.json_to_netcdf import *
# from cmct.shapefile_utils import *

# Force initial garbage collection
gc.collect()


40

In [2]:
# Reload modules to pick up any changes to imports
import importlib
import cmct.calving
import cmct.calving_modules.residual_calculation
from cmct.calving_modules.plotting_utils import *
from cmct.calving import calculate_basin_statistics, format_basin_stats
from cmct.calving import calculate_basin_statistics

importlib.reload(cmct.calving)
importlib.reload(cmct.calving_modules.residual_calculation)


# Re-import to ensure functions are available
from cmct.calving import *


In [3]:
# Observation Dataset
# Ice sheet
loc = "GIS"  # 'GIS' or 'AIS'

# Set the observation data dir path
obs_filename = cmct_dir + "/data/calving/observed_icemask_ismip_annual.nc"

# To use aggregation functions for basin
basin_aggregation = True  # IMPORTANT

basin_filename = (
    cmct_dir + "/bin/Calving/GRE_Basins_IMBIE2_v1.3/GRE_Basins_IMBIE2_v1.3.shp"
)

# Set the Model Data dir path
# model_filename = cmct_dir + "/test/calving/ensemble/sftgif_B001_hist.nc"
model_files = cmct_dir + "/test/calving/ensemble/*.nc"

# Set time range for comparison
start_year = 2007
end_year = 2010

# List of basins (ex ["NW", "NE"]) to compare if all -> "all", if none -> False
# If you do not know which basins are in the model, you can put "auto"
# NOTE: Align this list with the basins in the model.
basin_list = ["NW"]

# Output filetype and filename
filetype = "netcdf"  # netcdf or json or None
filename = "calving_comparison"

# Optional Configurations
interpolation_method = "slinear"  # 'nearest', 'linear', 'cubic'
accuracy_calculation_method = "mean"  # 'mean', 'RMS',

colors = {
    "CW": "blue",
    "NE": "red",
    "SE": "green",
    "SW": "orange",
    "NO": "purple",
    "NW": "brown",
}


In [4]:
# Check if observation file exist
if not os.path.exists(obs_filename):
    raise FileNotFoundError(f"Observation file not found: {obs_filename}")

if basin_aggregation and not os.path.exists(basin_filename):
    raise FileNotFoundError(f"Basin shapefile not found: {basin_filename}")
    # Load basin shapes

print(basin_filename)
basins, basin_list = load_basins(basin_filename, basin_list)

print(obs_filename)
gsfc = load_gsfc_calving(obs_filename, basins)
gsfc.ds["time"] = standardising_time_var(gsfc.time)




/Users/aditya_pachpande/Documents/GitHub/CmCt/bin/Calving/GRE_Basins_IMBIE2_v1.3/GRE_Basins_IMBIE2_v1.3.shp
/Users/aditya_pachpande/Documents/GitHub/CmCt/data/calving/observed_icemask_ismip_annual.nc


In [5]:
model_files = glob.glob(model_files)
# Sort the files for consistent ordering
model_files.sort()
# Generate model names from filenames (extract basename without extension)
model_names = [os.path.splitext(os.path.basename(f))[0] for f in model_files]

# Validate that we found model files
if not model_files:
    raise FileNotFoundError(
        f"No model files found matching pattern: {model_filename_template}"
    )

print(f"Found {len(model_files)} model files:")
for file in model_files:
    print(os.path.basename(file))

Found 5 model files:
sftgif_B001_hist.nc
sftgif_B002_hist.nc
sftgif_B003_hist.nc
sftgif_B004_hist.nc
sftgif_B005_hist.nc


In [9]:
def process_single_model(file):
    print(file)
    model_res = load_model_calving(file)
    model_res.ds["time"] = standardising_time_var(model_res.time)

    # Handelling Time Range
    checking_calving_daterange(
        gsfc.time.values, model_res.time.values, start_year, end_year
    )
    interpolater = Interpolater(model_res, gsfc)
    model_res.ds = interpolater.interpolate()
    print(f"\nResampled data shape: {model_res.ds.dims}")
    
    years = np.arange(start_year, end_year + 1)
    
    residuals_dataset = create_calving_dataset(gsfc, model_res, years, basins)
    residuals = load_residuals(residuals_dataset)
    basin_stats = calculate_basin_statistics(residuals)

    print(format_basin_stats(basin_stats))
    
    del model_res, interpolater, residuals_dataset, residuals
    gc.collect()
    
    return basin_stats
    


In [10]:
basin_stats_array = []
for i in model_files:
    try:
        if not os.path.exists(i):
            raise FileNotFoundError(f"Model file not found: {i}")
        print(f"Processing model file: {i}")
        basin_stats = process_single_model(i)
        basin_stats_array.append(basin_stats)
        
        del basin_stats
        gc.collect()
        
    except Exception as e:
        print(f"Error checking model file {i}: {e}")
        continue

Processing model file: /Users/aditya_pachpande/Documents/GitHub/CmCt/test/calving/ensemble/sftgif_B001_hist.nc
/Users/aditya_pachpande/Documents/GitHub/CmCt/test/calving/ensemble/sftgif_B001_hist.nc
The selected dates 2007 to 2010 are within the overlapping data range.

Resampled data shape: FrozenMappingWarningOnValuesAccess({'time': 14, 'x': 1680, 'y': 2880})
The selected dates 2007 to 2010 are within the overlapping data range.

Resampled data shape: FrozenMappingWarningOnValuesAccess({'time': 14, 'x': 1680, 'y': 2880})


2025-07-17 21:30:31,359 - ERROR - Basin mask shape (1680, 2880) doesn't match expected (2880, 1680)
2025-07-17 21:30:31,360 - ERROR - This indicates the coordinate transformation didn't work as expected
2025-07-17 21:30:31,360 - ERROR - This indicates the coordinate transformation didn't work as expected


=== Statistics for Year 2007 ===
Basin | Sum       | Count    | Mean        | Winsorized  | Outlier Wgt | Std        | RMS
-------------------------------------------------------------------------------------
NW    | 415.95709228515625 |   270870 |  0.00153563 |  0.00158687 |  0.00001312 |   0.056317 |   0.056338
-------------------------------------------------------------------------------------


=== Statistics for Year 2008 ===
Basin | Sum       | Count    | Mean        | Winsorized  | Outlier Wgt | Std        | RMS
-------------------------------------------------------------------------------------
NW    | 448.8782653808594 |   270870 |  0.00165717 |  0.00171128 |  0.00001562 |   0.056234 |   0.056258
-------------------------------------------------------------------------------------


=== Statistics for Year 2009 ===
Basin | Sum       | Count    | Mean        | Winsorized  | Outlier Wgt | Std        | RMS
------------------------------------------------------------------------

2025-07-17 21:30:37,428 - ERROR - Basin mask shape (1680, 2880) doesn't match expected (2880, 1680)
2025-07-17 21:30:37,428 - ERROR - This indicates the coordinate transformation didn't work as expected
2025-07-17 21:30:37,428 - ERROR - This indicates the coordinate transformation didn't work as expected


=== Statistics for Year 2007 ===
Basin | Sum       | Count    | Mean        | Winsorized  | Outlier Wgt | Std        | RMS
-------------------------------------------------------------------------------------
NW    | 415.95709228515625 |   270870 |  0.00153563 |  0.00158687 |  0.00001312 |   0.056317 |   0.056338
-------------------------------------------------------------------------------------


=== Statistics for Year 2008 ===
Basin | Sum       | Count    | Mean        | Winsorized  | Outlier Wgt | Std        | RMS
-------------------------------------------------------------------------------------
NW    | 448.8782653808594 |   270870 |  0.00165717 |  0.00171128 |  0.00001562 |   0.056234 |   0.056258
-------------------------------------------------------------------------------------


=== Statistics for Year 2009 ===
Basin | Sum       | Count    | Mean        | Winsorized  | Outlier Wgt | Std        | RMS
------------------------------------------------------------------------

...

=== Statistics for Year 2007 ===
Basin | Sum       | Count    | Mean        | Winsorized  | Outlier Wgt | Std        | RMS
-------------------------------------------------------------------------------------
NW    | 415.95709228515625 |   270870 |  0.00153563 |  0.00158687 |  0.00001312 |   0.056317 |   0.056338
-------------------------------------------------------------------------------------


=== Statistics for Year 2008 ===
Basin | Sum       | Count    | Mean        | Winsorized  | Outlier Wgt | Std        | RMS
-------------------------------------------------------------------------------------
NW    | 448.8782653808594 |   270870 |  0.00165717 |  0.00171128 |  0.00001562 |   0.056234 |   0.056258
-------------------------------------------------------------------------------------


=== Statistics for Year 2009 ===
Basin | Sum       | Count    | Mean        | Winsorized  | Outlier Wgt | Std        | RMS
------------------------------------------------------------------------

2025-07-17 21:30:49,569 - ERROR - Basin mask shape (1680, 2880) doesn't match expected (2880, 1680)
2025-07-17 21:30:49,570 - ERROR - This indicates the coordinate transformation didn't work as expected
2025-07-17 21:30:49,570 - ERROR - This indicates the coordinate transformation didn't work as expected


=== Statistics for Year 2007 ===
Basin | Sum       | Count    | Mean        | Winsorized  | Outlier Wgt | Std        | RMS
-------------------------------------------------------------------------------------
NW    | 415.95709228515625 |   270870 |  0.00153563 |  0.00158687 |  0.00001312 |   0.056317 |   0.056338
-------------------------------------------------------------------------------------


=== Statistics for Year 2008 ===
Basin | Sum       | Count    | Mean        | Winsorized  | Outlier Wgt | Std        | RMS
-------------------------------------------------------------------------------------
NW    | 448.8782653808594 |   270870 |  0.00165717 |  0.00171128 |  0.00001562 |   0.056234 |   0.056258
-------------------------------------------------------------------------------------


=== Statistics for Year 2009 ===
Basin | Sum       | Count    | Mean        | Winsorized  | Outlier Wgt | Std        | RMS
------------------------------------------------------------------------

2025-07-17 21:30:55,639 - ERROR - Basin mask shape (1680, 2880) doesn't match expected (2880, 1680)
2025-07-17 21:30:55,639 - ERROR - This indicates the coordinate transformation didn't work as expected
2025-07-17 21:30:55,639 - ERROR - This indicates the coordinate transformation didn't work as expected


=== Statistics for Year 2007 ===
Basin | Sum       | Count    | Mean        | Winsorized  | Outlier Wgt | Std        | RMS
-------------------------------------------------------------------------------------
NW    | 415.95709228515625 |   270870 |  0.00153563 |  0.00158687 |  0.00001312 |   0.056317 |   0.056338
-------------------------------------------------------------------------------------


=== Statistics for Year 2008 ===
Basin | Sum       | Count    | Mean        | Winsorized  | Outlier Wgt | Std        | RMS
-------------------------------------------------------------------------------------
NW    | 448.8782653808594 |   270870 |  0.00165717 |  0.00171128 |  0.00001562 |   0.056234 |   0.056258
-------------------------------------------------------------------------------------


=== Statistics for Year 2009 ===
Basin | Sum       | Count    | Mean        | Winsorized  | Outlier Wgt | Std        | RMS
------------------------------------------------------------------------

In [12]:
print(f"Successfully processed {len(basin_stats_array)} model files")
print(f"Basin stats array length: {len(basin_stats_array)}")
if basin_stats_array:
    print(f"First model stats keys: {list(basin_stats_array[0].keys())}")
    print(f"Sample basin stats for first model: {basin_stats_array[0]}")

Successfully processed 5 model files
Basin stats array length: 5
First model stats keys: [np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010)]
Sample basin stats for first model: {np.int64(2007): {np.str_('NW'): {'count': 270870, 'mean': np.float32(0.0015356337), 'std': np.float32(0.0563172), 'min': np.float32(-1.0), 'max': np.float32(1.0), 'rms': np.float32(0.056338128), 'rss': np.float32(859.73724), 'sum': np.float32(415.9571), 'winsorized_mean': np.float64(0.0015868740301593946), 'outlier_weighted_mean': np.float32(1.3117617e-05)}}, np.int64(2008): {np.str_('NW'): {'count': 270870, 'mean': np.float32(0.0016571723), 'std': np.float32(0.056233827), 'min': np.float32(-1.0), 'max': np.float32(1.0), 'rms': np.float32(0.056258246), 'rss': np.float32(857.3009), 'sum': np.float32(448.87827), 'winsorized_mean': np.float64(0.0017112816254955842), 'outlier_weighted_mean': np.float32(1.562134e-05)}}, np.int64(2009): {np.str_('NW'): {'count': 270870, 'mean': np.float32(0.001719219), '

# Ensemble Plotting and Analysis

In [15]:
# Reload the plotting module to ensure latest changes are available
import importlib
import cmct.calving_modules.plotting_utils
importlib.reload(cmct.calving_modules.plotting_utils)

# Import ensemble plotting utilities
from cmct.calving_modules.plotting_utils import (
    create_ensemble_time_series_plot,
    create_interactive_ensemble_plot,
    create_ensemble_statistics_summary
)

In [ ]:
# Create interactive ensemble plot with dropdown for statistic selection
print("Creating interactive ensemble time series plot...")
print(f"Models: {model_names}")
print(f"Basins: {basin_list}")

# Create the interactive plot
ensemble_plot_widget = create_interactive_ensemble_plot(
    basin_stats_array, 
    model_names, 
    basin_list=basin_list
)

# Display the widget
ensemble_plot_widget

Creating interactive ensemble time series plot...
Models: ['sftgif_B001_hist', 'sftgif_B002_hist', 'sftgif_B003_hist', 'sftgif_B004_hist', 'sftgif_B005_hist']
Basins: ['NW']
